In [32]:
pip install -r requirements.txt


In [33]:
import os
import math
import pickle
from typing import List, Union

import pandas as pd
from tqdm import tqdm
from groq import Groq
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from getpass import getpass

In [34]:
# ========= Config ========= #

os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API Key: ")

INDEX_DIR = "faiss_index_bioasq_full"
STATE_FILE = "checkpoint_all_chunks.pkl"
BATCH_SIZE = 3000
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_NAME = "llama3-8b-8192"

Enter your GROQ API Key: ··········


In [35]:
# ========= Data Loading ========= #
def load_data():
    passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
    test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")
    return passages, test


In [36]:
# ========= Text Processing ========= #
def chunk_documents(passages: pd.DataFrame, chunk_size=1000, chunk_overlap=100) -> List[Document]:
    docs = [Document(page_content=text) for text in passages["passage"].dropna()]
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)

In [37]:
# ========= FAISS Indexing ========= #
def build_faiss_index(chunks, index_dir=INDEX_DIR, state_file=STATE_FILE) -> FAISS:
    total_chunks = len(chunks)
    num_batches = math.ceil(total_chunks / BATCH_SIZE)
    embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, encode_kwargs={"batch_size": 64})

    if os.path.exists(state_file) and os.path.exists(index_dir):
        with open(state_file, "rb") as f:
            start_batch = pickle.load(f)
        faiss_index = FAISS.load_local(index_dir, embedding_model, allow_dangerous_deserialization=True)
        print(f"Resuming from batch {start_batch}")
    else:
        start_batch = 0
        faiss_index = None
        print("Starting fresh FAISS index...")

    for i in tqdm(range(start_batch, num_batches), desc="Embedding batches"):
        batch_chunks = chunks[i * BATCH_SIZE: (i + 1) * BATCH_SIZE]
        if faiss_index is None:
            faiss_index = FAISS.from_documents(batch_chunks, embedding_model)
        else:
            faiss_index.add_documents(batch_chunks)

        faiss_index.save_local(index_dir)
        with open(state_file, "wb") as f:
            pickle.dump(i + 1, f)

    print("Index ready.")
    return faiss_index

In [38]:
# ========= Load Model and Chain ========= #
def get_rag_chain(index_dir=INDEX_DIR) -> RetrievalQA:
    embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, encode_kwargs={"batch_size": 64})
    retriever = FAISS.load_local(index_dir, embedding_model, allow_dangerous_deserialization=True).as_retriever(search_kwargs={"k": 10})

    llm = ChatGroq(api_key=os.environ["GROQ_API_KEY"], model_name=MODEL_NAME)

    prompt_template = PromptTemplate.from_template("""
    You are a biomedical expert AI. Based on the provided documents, answer the question concisely.
    If the answer is not explicitly stated, do not make assumptions.

    Context:
    {context}

    Question: {question}
    Answer:
    """)

    return RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt_template}
    )


In [39]:
def evaluate_rag(rag_chain: RetrievalQA, test_queries: List[dict]):
    """
    Evaluates a RetrievalQA RAG chain using provided test queries.

    Args:
        rag_chain (RetrievalQA): The RetrievalQA chain.
        test_queries (List[dict]): A list of test cases with 'question' and 'expected' keys.

    Returns:
        None
    """
    correct = 0
    print("📋 List of Test Queries:")
    for i, test in enumerate(test_queries, 1):
        print(f"{i}. {test['question']}")
    print("\nRunning evaluation...\n")

    for i, test in enumerate(test_queries, 1):
        # Get the result using .invoke and extract 'result'
        output = rag_chain.invoke({"query": test["question"]})
        result = output["result"].lower()
        expected = test["expected"]

        # Evaluation logic
        is_correct = False
        if isinstance(expected, set):
            matched = {term for term in expected if term.lower() in result}
            is_correct = len(matched) >= 4
        else:
            is_correct = expected.lower() in result

        correct += int(is_correct)

        # Print result details
        print(f"Q{i}: {test['question']}")
        print(f"Expected: {expected}")
        print(f"Predicted: {result}")
        print(f"Correct: {'Yes ✅' if is_correct else 'No ❌'}\n")

    # Final accuracy
    accuracy = correct / len(test_queries) * 100
    print(f"Final Accuracy: {correct}/{len(test_queries)} = {accuracy:.2f}%")


In [40]:
# ========= Run Pipeline ========= #
def main():
    df_passages, df_test = load_data()
    chunks = chunk_documents(df_passages)
    print("Total Chunks:", len(chunks))

    build_faiss_index(chunks)

    rag_chain = get_rag_chain()

    test_queries = [
        {"question": "Is Hirschsprung disease a mendelian or a multifactorial disorder?", "expected": "multifactorial"},
        {"question": "List signaling molecules (ligands) that interact with the receptor EGFR?", "expected": {"EGF", "TGF-α", "AREG", "EPR", "HB-EGF", "BTC", "EPG"}},
        {"question": "Is the protein Papilin secreted?", "expected": "yes"},
        {"question": "Is RANKL secreted from the cells?", "expected": "yes"},
        {"question": "Are long non-coding RNAs spliced?", "expected": "yes"}
    ]

    evaluate_rag(rag_chain, test_queries)


if __name__ == "__main__":
    main()

Total Chunks: 69217
Resuming from batch 24


Embedding batches: 0it [00:00, ?it/s]

Index ready.


📋 List of Test Queries:
1. Is Hirschsprung disease a mendelian or a multifactorial disorder?
2. List signaling molecules (ligands) that interact with the receptor EGFR?
3. Is the protein Papilin secreted?
4. Is RANKL secreted from the cells?
5. Are long non-coding RNAs spliced?

Running evaluation...

Q1: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Expected: multifactorial
Predicted: based on the provided documents, hirschsprung disease is a complex and multifactorial disorder. while the genetics of hirschsprung's disease are highly complex, and the majority of known genetic sites relating to the main susceptibility pathways (ret and ednrb), the non-syndromic and familial forms of the disease have complex patterns of inheritance, which are not strictly mendelian. the findings suggest that low-penetrance mutations would be necessary but not sufficient, and the additional presence of the "hirschsprung disease haplotype" could contribute to the manifestation of the d